In [ ]:
import os 
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
from transformers import AutoTokenizer,AutoConfig, AutoModel

## config查看模型配置

In [ ]:
model="lxyuan/distilbert-base-multilingual-cased-sentiments-student"
config = AutoConfig.from_pretrained(model,local_files_only=True)
config

In [ ]:
Model = AutoModel.from_pretrained(model)
Model.config

In [ ]:
model_name = "agentlans/multilingual-e5-small-aligned-sentiment"
MODEL = AutoModel.from_pretrained(model_name)

In [ ]:
from transformers import AutoModelForSequenceClassification
model="lxyuan/distilbert-base-multilingual-cased-sentiments-student"
model = AutoModelForSequenceClassification.from_pretrained(model)
model.config.num_labels

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("./token_save")
text = "这部电影真好看"
tokens = tokenizer(text,return_tensors = "pt")
tokens

In [ ]:
model(**tokens)

## 加载Model Head

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model,local_files_only = True, num_labels = 10,ignore_mismatched_sizes=True)
model.config.num_labels     ## 注意！ignore_mismatched_sizes=True

## 实践 

In [ ]:
import os 
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
from transformers import AutoTokenizer,AutoConfig, AutoModel
import pandas as pd 

In [ ]:
data = pd.read_csv("./ChnSentiCorp_htl_all.csv")
data = data.dropna()    
data.head()

In [ ]:
from torch.utils.data import Dataset,random_split
class review_dataset(Dataset):
    def __init__(self):
        super().__init__()
        self.data = pd.read_csv("./ChnSentiCorp_htl_all.csv").dropna()
    def __getitem__(self, index):
        return self.data.iloc[index]["review"],self.data.iloc[index]["label"]
    def __len__(self):
        return len(self.data)
review_dataset = review_dataset()
train_set, valid_set = random_split(review_dataset,lengths=[0.8,0.2])

In [ ]:
import torch 
model_name="lxyuan/distilbert-base-multilingual-cased-sentiments-student"
tokenizer = AutoTokenizer.from_pretrained(model_name,local_file_only = True)
def collate_func(batch):
    texts, labels = [],[]
    for item in batch:
        texts.append(item[0])
        labels.append(item[1])
    inputs = tokenizer(texts, max_length = 64, padding =True, truncation = True,return_tensors="pt")
    inputs["labels"] = torch.tensor(labels)
    return inputs    

In [ ]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_set,batch_size=64,shuffle=True,collate_fn=collate_func)
valid_loader = DataLoader(valid_set,batch_size=64,shuffle=False,collate_fn=collate_func)

In [ ]:
from torch.optim import Adam
from transformers import AutoModelForSequenceClassification
Model = AutoModelForSequenceClassification.from_pretrained(model_name)
optimizer = Adam(Model.parameters(), lr=2e-5)

In [ ]:
config = AutoConfig.from_pretrained(model_name)
config

In [ ]:
def accuracy():
    Model.eval()
    acc_num = 0 
    with torch.inference_mode():    
        for batch in valid_loader:
            outputs = Model(**batch)
            pred = torch.argmax(outputs.logits, dim=1)
            acc_num += (pred.long()== batch['label'].long()).float().sum()
    return acc_num / len(valid_set)


def train(epoch =1):
    Model.train()
    global_step = 0
    for ep in range(epoch):
        for batch in train_loader :
            optimizer.zero_grad()
            outputs = Model(**batch)
            outputs.loss.backward()
            optimizer.step()
            if global_step %10 ==0: 
                print(f"ep:{ep}, global_step:{global_step}, loss:{outputs.loss.item():.4f}")
            global_step += 1
        accuracy = accuracy()
        print(f"accuracy:{accuracy}.3f")

In [ ]:
train()

In [ ]:
from transformers import pipeline
pipe = pipeline("text-classification",model = model_name, tokenizer = tokenizer)
text = "这家饭店不错！"
res = pipe(text)


In [ ]:
res